In [1]:
import tensorflow as tf
import glob
import os, re
import numpy as np
from sklearn.model_selection import train_test_split

print(tf.__version__)

# ==========================================
# 1. 데이터 읽어오기 (윈도우 환경에 맞게 경로 수정)
# ==========================================
# 사장님 PC 경로로 수정했습니다.
txt_file_path = r'D:\AiSon\DTS\Lyrics\*.txt'

txt_list = glob.glob(txt_file_path)

raw_corpus = []

# 여러개의 txt 파일을 모두 읽어서 raw_corpus 에 담습니다.
for txt_file in txt_list:
    # 윈도우 호환을 위해 encoding 추가
    try:
        with open(txt_file, "r", encoding='utf-8') as f:
            raw = f.read().splitlines()
            raw_corpus.extend(raw)
    except:
        pass # 에러나면 패스

print("데이터 크기:", len(raw_corpus))
print("Examples:\n", raw_corpus[:3])

# ==========================================
# [중요] 모델이 학습하려면 전처리와 토큰화가 반드시 필요합니다.
# (이 부분이 없으면 generate_text를 실행할 수 없습니다)
# ==========================================
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,¿]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<start> ' + sentence + ' <end>'
    return sentence

corpus = []
for sentence in raw_corpus:
    if len(sentence) == 0: continue
    if sentence[-1] == ":": continue
    preprocessed_sentence = preprocess_sentence(sentence)
    if len(preprocessed_sentence.split()) <= 15: # 길이 제한
        corpus.append(preprocessed_sentence)

tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=12000, filters=' ', oov_token="<unk>"
)
tokenizer.fit_on_texts(corpus)
tensor = tokenizer.texts_to_sequences(corpus)
tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post', maxlen=15)

# ==========================================
# 2. 데이터셋 분리 (여기가 바로 <코드 작성> 정답 부분!)
# ==========================================
src_input = tensor[:, :-1]
tgt_input = tensor[:, 1:]

# <코드 작성> 부분을 채웠습니다.
enc_train, enc_val, dec_train, dec_val = train_test_split(
    src_input, 
    tgt_input, 
    test_size=0.2, 
    random_state=2026
)

print(f"Train: {enc_train.shape}, Val: {enc_val.shape}")

# ==========================================
# 3. 모델 구성 및 학습 (lyricist 정의)
# ==========================================
class TextGenerator(tf.keras.Model):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_size)
        self.rnn_1 = tf.keras.layers.LSTM(hidden_size, return_sequences=True)
        self.rnn_2 = tf.keras.layers.LSTM(hidden_size, return_sequences=True)
        self.linear = tf.keras.layers.Dense(vocab_size)
        
    def call(self, x):
        out = self.embedding(x)
        out = self.rnn_1(out)
        out = self.rnn_2(out)
        out = self.linear(out)
        return out

embedding_size = 256
hidden_size = 1024
lyricist = TextGenerator(tokenizer.num_words + 1, embedding_size , hidden_size)

optimizer = tf.keras.optimizers.Adam()
loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none'
)

lyricist.compile(loss=loss, optimizer=optimizer)

# 데이터셋 배칭
BATCH_SIZE = 256
train_dataset = tf.data.Dataset.from_tensor_slices((enc_train, dec_train)).shuffle(len(enc_train)).batch(BATCH_SIZE, drop_remainder=True)
val_dataset = tf.data.Dataset.from_tensor_slices((enc_val, dec_val)).batch(BATCH_SIZE, drop_remainder=True)

# 학습 (시간 관계상 10번만 빠르게)
lyricist.fit(train_dataset, epochs=10, validation_data=val_dataset)

# ==========================================
# 4. 가사 생성 함수 (제출용)
# ==========================================
def generate_text(model, tokenizer, init_sentence="<start>", max_len=20):
    test_input = tokenizer.texts_to_sequences([init_sentence])
    test_tensor = tf.convert_to_tensor(test_input, dtype=tf.int64)
    end_token = tokenizer.word_index["<end>"]

    while True:
        predict = model(test_tensor) 
        predict_word = tf.argmax(tf.nn.softmax(predict, axis=-1), axis=-1)[:, -1] 
        test_tensor = tf.concat([test_tensor, tf.expand_dims(predict_word, axis=0)], axis=-1)
        if predict_word.numpy()[0] == end_token: break
        if test_tensor.shape[1] >= max_len: break

    generated = ""
    for word_index in test_tensor[0].numpy():
        generated += tokenizer.index_word[word_index] + " "
    return generated

# ==========================================
# [최종 제출] 모델이 생성한 가사 한 줄 출력
# ==========================================
print("\n👇 [제출할 가사 한 줄] 👇")
print(generate_text(lyricist, tokenizer, init_sentence="<start> i love", max_len=20))

2.20.0
데이터 크기: 20001
Examples:
 ['when i find myself in times of trouble mother mary comes to me', 'speaking words of wisdom let it be', 'and in my hour of darkness she is standing right in front of me']
Train: (14401, 14), Val: (3601, 14)
Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - loss: 5.0170 - val_loss: 2.5521
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 120s 2s/step - loss: 2.3063 - val_loss: 1.8066
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - loss: 1.6454 - val_loss: 1.1162
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - loss: 0.9390 - val_loss: 0.5298
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - loss: 0.4432 - val_loss: 0.2898
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step - loss: 0.2701 - val_loss: 0.2374
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - loss: 0.2325 - val_loss: 0.2218
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step - loss: 0.2205 - val_loss: 0.2157
Epoch 9/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - loss: 0.2155 - val_loss: 0.